# Module 05 - Embeddings and Positions

Use this notebook as the working space for the Module 05 exercises. Keep `notebooks/clean/` pristine; work in the copy created under `notebooks/solutions/`.

In [ ]:
from __future__ import annotations

from pathlib import Path

import matplotlib.pyplot as plt
import torch

from g2c.embeddings import (
    LearnedPositionalEmbedding,
    RotaryEmbedding,
    SinusoidalPositionalEmbedding,
    TokenEmbedding,
)
from g2c.nn import CrossEntropyLoss, Linear, SGD
from g2c.tokenizer import BPETokenizer

torch.manual_seed(0)

## Before the Notebook

Use the tests to implement the library pieces first. The notebook assumes `TokenEmbedding`, additive positional embeddings, and `RotaryEmbedding` become available as you progress through `tests/test_embeddings.py`.

In [ ]:
"Run from the terminal: .venv/bin/python -m pytest tests/test_embeddings.py -x"
"Question: Which embeddings test is the next one failing, and what implementation does it point at?"
"Answer: "

## Exercise 1 - Token and Learned Position Lookups

The core embedding operation is integer indexing into a learned table. Predict the shapes first, then use the checks below after the forwards are implemented.

In [ ]:
"Question: If ids has shape (B, T) and the embedding table has shape (V, C), what shape should weight[ids] have?"
"Answer: "

"Question: Why is the token embedding forward pass just table indexing, not matrix multiplication?"
"Answer: "

"Question: LearnedPositionalEmbedding returns shape (T, C). Why can it be added to token vectors shaped (B, T, C)?"
"Answer: "

In [ ]:
token_emb = TokenEmbedding(vocab_size=12, embedding_dim=4)
ids = torch.tensor([[3, 1, 7], [0, 5, 5]])
learned_pos = LearnedPositionalEmbedding(max_seq_len=16, embedding_dim=4)

# Uncomment after TokenEmbedding.forward and LearnedPositionalEmbedding.forward work.
# token_vectors = token_emb(ids)
# pos_vectors = learned_pos(seq_len=ids.shape[1])
# combined = token_vectors + pos_vectors
#
# print("ids:", tuple(ids.shape))
# print("token vectors:", tuple(token_vectors.shape))
# print("position vectors:", tuple(pos_vectors.shape))
# print("combined:", tuple(combined.shape))
# assert token_vectors.shape == (2, 3, 4)
# assert pos_vectors.shape == (3, 4)
# assert combined.shape == (2, 3, 4)
# assert torch.allclose(token_vectors[0, 0], token_emb.weight[3])

## Exercise 2 - Sinusoidal Positional Encoding

Build the fixed sine/cosine table in `SinusoidalPositionalEmbedding.__init__`, then inspect the values and the heatmap.

In [ ]:
"Question: At position 0, what should every sine slot contain? What should every cosine slot contain?"
"Answer: "

"Question: Why does this implementation require an even embedding_dim?"
"Answer: "

"Question: Why should the sinusoidal table have no learnable parameters?"
"Answer: "

In [ ]:
# Uncomment after SinusoidalPositionalEmbedding.__init__ and forward work.
# sinusoidal = SinusoidalPositionalEmbedding(max_seq_len=64, embedding_dim=32)
# prefix = sinusoidal(seq_len=8)
# print("full table:", tuple(sinusoidal.weight.shape))
# print("prefix:", tuple(prefix.shape))
# print("position 0:", sinusoidal.weight[0, :8])
# assert torch.allclose(sinusoidal.weight[0, 0::2], torch.zeros(16), atol=1e-6)
# assert torch.allclose(sinusoidal.weight[0, 1::2], torch.ones(16), atol=1e-6)
#
# plt.figure(figsize=(9, 4))
# plt.imshow(sinusoidal.weight.T, aspect="auto", interpolation="nearest")
# plt.xlabel("position")
# plt.ylabel("dimension")
# plt.title("Sinusoidal positional encoding")
# plt.colorbar()
# plt.show()

## Exercise 3 - RoPE Table Construction

`RotaryEmbedding.__init__` precomputes cosine and sine tables. The split-halves convention is what makes those tables align with `_rotate_half`.

In [ ]:
"Question: In the split-halves RoPE convention, which dimensions are paired together when embedding_dim is 8?"
"Answer: "

"Question: Why do the cos and sin tables both have shape (max_seq_len, embedding_dim), not (max_seq_len, embedding_dim // 2)?"
"Answer: "

"Question: What should cos[0] and sin[0] contain, and why?"
"Answer: "

In [ ]:
# Uncomment after RotaryEmbedding.__init__ works.
# rope = RotaryEmbedding(max_seq_len=32, embedding_dim=8)
# print("cos:", tuple(rope.cos.shape))
# print("sin:", tuple(rope.sin.shape))
# print("cos[0]:", rope.cos[0])
# print("sin[0]:", rope.sin[0])
# assert torch.allclose(rope.cos[0], torch.ones(8), atol=1e-6)
# assert torch.allclose(rope.sin[0], torch.zeros(8), atol=1e-6)

## Exercise 4 - Apply RoPE

Once the tables exist, the forward pass is a rotation: `x * cos + rotate_half(x) * sin`. The important check is that dot products depend on relative position, not absolute position.

In [ ]:
"Question: Why does a rotation preserve the L2 norm of each vector?"
"Answer: "

"Question: Why is position 0 the identity rotation?"
"Answer: "

"Question: In attention, why is a relative-position dot-product property more useful than encoding only absolute positions?"
"Answer: "

In [ ]:
def rotated_dot(rope: RotaryEmbedding, q: torch.Tensor, k: torch.Tensor, m: int, n: int) -> float:
    """Return dot(R(q, m), R(k, n)) for one absolute position pair."""
    seq_len = max(m, n) + 1
    seq_q = torch.zeros(1, seq_len, q.shape[-1])
    seq_k = torch.zeros(1, seq_len, k.shape[-1])
    seq_q[0, m] = q
    seq_k[0, n] = k
    return (rope(seq_q)[0, m] * rope(seq_k)[0, n]).sum().item()


# Uncomment after RotaryEmbedding.forward works.
# torch.manual_seed(0)
# rope = RotaryEmbedding(max_seq_len=20, embedding_dim=16)
# q = torch.randn(16)
# k = torch.randn(16)
# same_offset_scores = [
#     rotated_dot(rope, q, k, 0, 2),
#     rotated_dot(rope, q, k, 3, 5),
#     rotated_dot(rope, q, k, 7, 9),
# ]
# print(same_offset_scores)
# assert max(same_offset_scores) - min(same_offset_scores) < 1e-4

## Exercise 5 - Train Tiny Co-occurrence Embeddings

This is a small skip-gram style model: choose a center token and predict nearby context tokens. The tokenizer and plotting boilerplate are provided; you fill in pair generation and one training step.

In [ ]:
tiny_corpus = """
king queen prince princess palace throne crown royal
man woman child family mother father sister brother
paris france rome italy madrid spain berlin germany
cat dog kitten puppy animal pet fur tail
king queen royal palace crown throne
paris france capital city rome italy capital city
"""

tiny_tokenizer = BPETokenizer()

# Uncomment after your Module 04 tokenizer works.
# tiny_tokenizer.train(tiny_corpus, vocab_size=300)
# corpus_ids = tiny_tokenizer.encode(tiny_corpus)
# vocab_size = len(tiny_tokenizer.vocab)
# print("corpus tokens:", len(corpus_ids))
# print("vocab size:", vocab_size)
# print("first 40 ids:", corpus_ids[:40])

In [ ]:
"Question: In a skip-gram pair, which token is the input and which token is the target?"
"Answer: "

"Question: Why does predicting nearby tokens push co-occurring tokens toward useful embedding geometry?"
"Answer: "

"Question: Why should you expect a tiny corpus to show weaker structure than pretrained word vectors?"
"Answer: "

In [ ]:
def make_skipgram_pairs(ids: list[int], window: int = 2) -> tuple[torch.Tensor, torch.Tensor]:
    """Return center IDs and context target IDs for a skip-gram objective.

    For every position i, pair ids[i] with each neighboring token inside
    `window`, excluding i itself.

    Example with window=1:
      [10, 20, 30] -> centers [10, 20, 20, 30], contexts [20, 10, 30, 20]
    """
    centers: list[int] = []
    contexts: list[int] = []

    # TODO: loop over each center position i
    # TODO: loop over neighboring positions j in [i - window, i + window]
    # TODO: skip j == i and out-of-bounds positions
    # TODO: append ids[i] to centers and ids[j] to contexts
    # TODO: return two torch.long tensors
    raise NotImplementedError("build center/context token pairs")

In [ ]:
class SkipGramEmbeddingModel:
    """Minimal skip-gram model: center token -> embedding -> context logits."""

    def __init__(self, vocab_size: int, embedding_dim: int) -> None:
        self.embedding = TokenEmbedding(vocab_size, embedding_dim)
        self.output = Linear(embedding_dim, vocab_size)

    def parameters(self) -> list[torch.Tensor]:
        return list(self.embedding.parameters()) + list(self.output.parameters())

    def __call__(self, center_ids: torch.Tensor) -> torch.Tensor:
        center_vectors = self.embedding(center_ids)
        return self.output(center_vectors)


def train_skipgram(
    model: SkipGramEmbeddingModel,
    center_ids: torch.Tensor,
    context_ids: torch.Tensor,
    steps: int = 500,
    batch_size: int = 128,
    lr: float = 0.2,
) -> list[float]:
    """Train on random batches and return the loss curve."""
    loss_fn = CrossEntropyLoss()
    optimizer = SGD(model.parameters(), lr=lr)
    losses: list[float] = []

    def train_step() -> float:
        # TODO: sample random indices into center_ids/context_ids
        # TODO: select a batch of center IDs and target context IDs
        # TODO: zero gradients
        # TODO: compute logits = model(batch_centers)
        # TODO: compute cross-entropy loss against batch_contexts
        # TODO: backward, optimizer step, and return float(loss.item())
        raise NotImplementedError("fill in one skip-gram training step")

    for _ in range(steps):
        losses.append(train_step())

    return losses

In [ ]:
# Uncomment after make_skipgram_pairs and train_skipgram work.
# centers, contexts = make_skipgram_pairs(corpus_ids, window=2)
# skipgram = SkipGramEmbeddingModel(vocab_size=vocab_size, embedding_dim=16)
# skipgram_losses = train_skipgram(skipgram, centers, contexts, steps=600, batch_size=128, lr=0.2)
# print("first loss:", skipgram_losses[0])
# print("final loss:", skipgram_losses[-1])
# assert skipgram_losses[-1] < skipgram_losses[0]
#
# plt.plot(skipgram_losses)
# plt.xlabel("step")
# plt.ylabel("cross-entropy")
# plt.title("Tiny skip-gram training loss")
# plt.show()

In [ ]:
def project_rows_2d(weight: torch.Tensor) -> torch.Tensor:
    """Project embedding rows to 2D with a small PCA using torch.linalg.svd."""
    rows = weight.detach()
    rows = rows - rows.mean(dim=0, keepdim=True)
    _, _, vh = torch.linalg.svd(rows, full_matrices=False)
    return rows @ vh[:2].T


def decode_token_label(tokenizer: BPETokenizer, token_id: int) -> str:
    return tokenizer.vocab[token_id].decode("utf-8", errors="replace")


# Uncomment after training the skip-gram model.
# coords = project_rows_2d(skipgram.embedding.weight)
# learned_token_ids = list(range(256, min(vocab_size, 300)))
#
# plt.figure(figsize=(8, 6))
# plt.scatter(coords[learned_token_ids, 0], coords[learned_token_ids, 1], s=18)
# for token_id in learned_token_ids:
#     plt.text(
#         coords[token_id, 0].item(),
#         coords[token_id, 1].item(),
#         decode_token_label(tiny_tokenizer, token_id),
#         fontsize=8,
#     )
# plt.title("Tiny learned token embeddings, projected to 2D")
# plt.show()

## Exercise 6 - Pretrained Vector Analogies

Use a small local GloVe-style text file if you have one under `data/`. The helper loads only the requested words, so it does not need to keep the full file in memory.

In [ ]:
"Question: What vector expression should approximate queen in the classic analogy?"
"Answer: "

"Question: Why is cosine similarity usually better than raw dot product for nearest-neighbor lookup in pretrained embeddings?"
"Answer: "

"Question: If the pretrained analogy works but your tiny model does not, what does that say about corpus scale and training signal?"
"Answer: "

In [ ]:
def load_glove_subset(path: Path, words: set[str]) -> dict[str, torch.Tensor]:
    """Load only selected word vectors from a whitespace-separated GloVe file."""
    vectors: dict[str, torch.Tensor] = {}
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            pieces = line.rstrip().split(" ")
            word = pieces[0]
            if word in words:
                values = [float(x) for x in pieces[1:]]
                vectors[word] = torch.tensor(values, dtype=torch.float32)

    missing = sorted(words - vectors.keys())
    if missing:
        print("missing words:", missing)
    return vectors


def normalized(v: torch.Tensor) -> torch.Tensor:
    return v / v.norm().clamp_min(1e-12)


def nearest_by_cosine(
    query: torch.Tensor,
    vectors: dict[str, torch.Tensor],
    exclude: set[str] | None = None,
    top_k: int = 5,
) -> list[tuple[str, float]]:
    """Return nearest words to query by cosine similarity."""
    exclude = exclude or set()

    # TODO: normalize query
    # TODO: score every word not in exclude with cosine similarity
    # TODO: sort descending and return top_k (word, score) pairs
    raise NotImplementedError("implement nearest-neighbor lookup")


def analogy(a: str, b: str, c: str, vectors: dict[str, torch.Tensor], top_k: int = 5):
    """Return nearest words to a - b + c."""
    query = vectors[a] - vectors[b] + vectors[c]
    return nearest_by_cosine(query, vectors, exclude={a, b, c}, top_k=top_k)

In [ ]:
glove_path = Path("data/glove.6B.50d.txt")
analogy_words = {
    "king", "queen", "man", "woman", "prince", "princess",
    "paris", "france", "italy", "rome", "madrid", "spain",
    "berlin", "germany", "london", "england",
}

# Uncomment if you have a local GloVe file at glove_path.
# pretrained = load_glove_subset(glove_path, analogy_words)
# print("king - man + woman:", analogy("king", "man", "woman", pretrained))
# print("paris - france + italy:", analogy("paris", "france", "italy", pretrained))

## Exercise 7 - Compare Positional Schemes

Plot learned, sinusoidal, and RoPE tables side-by-side. This is mostly an observation exercise: learned starts noisy, sinusoidal/RoPE show multi-frequency structure.

In [ ]:
"Question: Which positional scheme has learnable parameters?"
"Answer: "

"Question: What visual pattern do sinusoidal and RoPE tables share?"
"Answer: "

"Question: Why is RoPE applied to queries and keys inside attention rather than simply added to token embeddings here?"
"Answer: "

In [ ]:
# Uncomment after learned, sinusoidal, and RoPE implementations work.
# max_seq_len = 64
# embedding_dim = 32
# learned = LearnedPositionalEmbedding(max_seq_len=max_seq_len, embedding_dim=embedding_dim)
# sinusoidal = SinusoidalPositionalEmbedding(max_seq_len=max_seq_len, embedding_dim=embedding_dim)
# rope = RotaryEmbedding(max_seq_len=max_seq_len, embedding_dim=embedding_dim)
#
# tables = [
#     ("learned", learned.weight.detach()),
#     ("sinusoidal", sinusoidal.weight.detach()),
#     ("RoPE cos", rope.cos.detach()),
# ]
# fig, axes = plt.subplots(1, 3, figsize=(14, 4), constrained_layout=True)
# for ax, (title, table) in zip(axes, tables):
#     im = ax.imshow(table.T, aspect="auto", interpolation="nearest")
#     ax.set_title(title)
#     ax.set_xlabel("position")
#     ax.set_ylabel("dimension")
#     fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
# plt.show()